# 사출성형 공정데이터 기반 품질불량 사전예측 및 검사 우선순위 결정

2026년 제6회 K-인공지능 제조데이터 분석 경진대회 (일반국민/대학원생 부문, 과제 1)

**목표**: 최종 품질검사 이전 시점에 `PassOrFail`을 예측하는 모델을 개발하고,
모델이 어떤 조건에서 정확하게 작동하고 어떤 조건에서 실패하는지 분석한다.

이 노트북은 서면평가 6개 지표(데이터 이해·진단 / AI 모델 개발 / 영향요인·오류분석 /
현장 활용방안 / 창의성·차별성 / 코드·재현성) 순서를 그대로 따라가며,
각 코드 셀 위에는 **무엇을 왜 썼는지 근거**를 함께 남긴다.

## 0. 환경 설정

**무엇을 썼나**: `pandas`, `numpy`(데이터 처리) / `matplotlib`, `seaborn`(시각화) /
`scikit-learn`(베이스라인·평가·분할) / `xgboost`(비교모델) / `shap`(설명가능성) /
`imblearn`(불균형 대응).

**왜 썼나**: 서면평가 2번 항목이 "베이스라인 포함 2개 이상 모델 비교"를 요구하므로
선형모델(로지스틱회귀)과 트리 앙상블(RandomForest, XGBoost)을 함께 준비한다.
3번 항목(영향요인·오류분석)을 위해 `shap`을, 데이터가 극단적으로 불균형(양성 1.4~2.1%)하므로
`imblearn`을 미리 준비한다.

**근거**: 표 형태의 정형 제조 공정데이터(24개 연속형 변수, 1천여 행)에는 딥러닝보다
트리 기반 GBM 계열이 일반적으로 더 안정적인 성능을 내고, SHAP과의 호환성도 좋아
"영향요인 분석" 요구사항을 만족시키기 쉽다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, RepeatedStratifiedKFold, cross_val_score,
)
from sklearn.metrics import make_scorer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, precision_recall_curve, roc_auc_score, average_precision_score,
    classification_report, confusion_matrix, PrecisionRecallDisplay,
)
from sklearn.calibration import CalibratedClassifierCV

import xgboost as xgb
import shap

RANDOM_STATE = 42  # 재현성 확보를 위해 전 과정에서 동일한 시드 고정 (6. 코드 및 재현성)
np.random.seed(RANDOM_STATE)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.family"] = "Malgun Gothic"   # 한글 라벨 깨짐 방지 (Windows 기본 폰트)
plt.rcParams["axes.unicode_minus"] = False

pd.set_option("display.max_columns", 50)

## 1. 데이터 이해 및 진단 (서면평가 1번, 15점)

**요구내용**: 데이터의 생산단위, 변수 의미, 시간·설비·제품 간 관계, 결측·중복·이상치·불균형을
분석하고 전처리 및 검증전략을 제시.

**무엇을 썼나**: `pandas`로 4개 CSV(라벨 2종 + 무라벨 2종)를 로드하고 shape/dtype/결측치를 점검.

**왜 이 순서로 하나**: 모델링 전에 반드시 "이 데이터가 무엇인지"를 먼저 확정해야
서면평가 1번 배점을 확보할 수 있고, 이후 전처리·분할 전략의 근거가 여기서 나온다.

In [ ]:
DATA_DIR = "."  # 노트북과 같은 폴더에 4개 CSV가 위치 (재현성: 상대경로 사용)

labeled_cn7 = pd.read_csv(f"{DATA_DIR}/moldset_labeled_cn7.csv", index_col=0)
labeled_rg3 = pd.read_csv(f"{DATA_DIR}/moldset_labeled_rg3.csv", index_col=0)
unlabeled_cn7 = pd.read_csv(f"{DATA_DIR}/moldset_unlabeled_cn7.csv", index_col=0)
unlabeled_rg3 = pd.read_csv(f"{DATA_DIR}/moldset_unlabeled_rg3.csv", index_col=0)

# 어느 사출성형기(설비)에서 나온 데이터인지 컬럼으로 명시
# -> CN7/RG3는 서로 다른 설비이므로, 설비 간 공정조건 차이가 있는지 반드시 확인해야 함
labeled_cn7["machine"] = "CN7"
labeled_rg3["machine"] = "RG3"
unlabeled_cn7["machine"] = "CN7"
unlabeled_rg3["machine"] = "RG3"

for name, df in [("labeled_cn7", labeled_cn7), ("labeled_rg3", labeled_rg3),
                  ("unlabeled_cn7", unlabeled_cn7), ("unlabeled_rg3", unlabeled_rg3)]:
    print(f"{name:16s} shape={df.shape}")

labeled_cn7.head()

In [ ]:
# 결측치 / 중복행 / 무분산(constant) 변수 점검
# 왜: 서면평가 1번 배점 항목에 "결측·중복·이상치"를 명시적으로 요구하므로
#     실제로 없더라도 "확인했다"는 근거를 남겨야 한다.
for name, df in [("labeled_cn7", labeled_cn7), ("labeled_rg3", labeled_rg3),
                  ("unlabeled_cn7", unlabeled_cn7), ("unlabeled_rg3", unlabeled_rg3)]:
    n_missing = df.isna().sum().sum()
    n_dup = df.drop(columns=["machine"]).duplicated().sum()
    print(f"{name:16s} 결측치 합계={n_missing:5d}   중복행={n_dup:4d}  (전체 {len(df)}행 중 {n_dup/len(df)*100:.1f}%)")

# 무분산 변수 점검: 표준편차가 0에 가까운 변수는 모델에 아무 정보도 주지 못함
_feat_cols_tmp = [c for c in labeled_cn7.columns if c not in ("PassOrFail", "machine")]
zero_var_cols = labeled_cn7[_feat_cols_tmp].std()[lambda s: s < 1e-8].index.tolist()
print(f"\n무분산(constant) 변수: {zero_var_cols}")

**진단 결과 (중요)**

1. **중복행이 라벨 데이터의 약 50%** (CN7 594/1211행, RG3 566/1182행)를 차지한다.
   같은 설정값으로 여러 사출(cycle)을 연속 생산하면 로그 요약값이 동일하게 남는 제조 현장의
   특성상 자연스러운 현상일 수 있으나, **train/test를 나누기 전에 제거하지 않으면 동일한
   행이 train과 test에 동시에 들어가는 데이터 누수(leakage)**가 발생해 성능이 과대평가된다.
   → 2번(전처리) 단계에서 분할 전에 중복 제거를 반드시 수행한다.
2. **`Clamp_Open_Position`은 표준편차 0인 상수(무분산) 변수**다. 모델에 아무 정보도 주지
   않으므로 학습에서 제외한다 (제외하지 않아도 트리 모델은 자동으로 무시하지만, 로지스틱회귀는
   불필요한 계수를 추정하므로 명시적으로 드롭하는 것이 안전하고 해석도 깔끔해진다).

In [ ]:
# 클래스 불균형 확인 + "이미 표준화된 데이터"임을 확인
# 왜: (1) 서면평가 1번이 "불균형 분석"을 명시적으로 요구
#     (2) 각 변수의 평균≈0, 표준편차≈1이면 원본 단위가 아니라 z-score 스케일링이 이미
#         적용된 것 -> 뒤에서 StandardScaler를 또 적용하면 안 된다는 중요한 전처리 근거가 됨
print("=== PassOrFail 클래스 비율 ===")
for name, df in [("CN7", labeled_cn7), ("RG3", labeled_rg3)]:
    vc = df["PassOrFail"].value_counts()
    ratio = vc.get(1, 0) / len(df) * 100
    print(f"{name}: 정상={vc.get(0,0)}, 불량={vc.get(1,0)}  (불량비율 {ratio:.2f}%)")

print("\n=== 변수별 평균/표준편차 (표준화 여부 확인, CN7 예시) ===")
feature_cols = [c for c in labeled_cn7.columns if c not in ("PassOrFail", "machine")]
display(labeled_cn7[feature_cols].agg(["mean", "std"]).T.round(2).head(8))

In [ ]:
# 변수 간 상관관계 (다중공선성 점검)
# 왜: Barrel_Temperature_1~6처럼 물리적으로 연동된 변수는 서로 강하게 상관될 가능성이 높음.
#     선형모델(로지스틱회귀) 계수 해석을 왜곡할 수 있으므로 트리 모델을 병행하는 근거가 되고,
#     추후 SHAP 해석 시 "상관된 변수군"으로 묶어 해석해야 한다는 점을 미리 파악해둔다.
corr = labeled_cn7[feature_cols].corr()
plt.figure(figsize=(11, 9))
sns.heatmap(corr, cmap="coolwarm", center=0, square=True, linewidths=0.3)
plt.title("CN7 변수 간 상관관계 히트맵")
plt.tight_layout()
plt.show()

high_corr = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack()
    .sort_values(key=abs, ascending=False)
)
print("절대상관계수 0.8 이상 변수쌍:")
print(high_corr[high_corr.abs() >= 0.8])

### CN7 vs RG3: 설비(제품) 간 관계 확인

**왜 확인하나**: 서면평가 1번이 "설비·제품 간 관계"를 명시적으로 요구한다.
두 설비가 서로 다른 분포를 가진다면, 두 설비 데이터를 하나로 합쳐 학습할지
설비별로 분리해서 학습할지를 결정해야 하고, 이는 2번 항목(모델 개발) 설계와
직결되는 중요한 판단이다.

**무엇을 썼나**: `unlabeled` 데이터(설비별 각 3만5천여 행, 라벨 데이터보다 훨씬 큼)를 이용해
변수별 분포를 비교한다. 표본 수가 충분히 크므로 평균 차이가 통계적으로도 실질적으로도
의미 있는지 함께 본다.

In [ ]:
from scipy.stats import ks_2samp

# 설비별 분포 차이: Kolmogorov-Smirnov 검정
# 왜 KS 검정: 두 표본의 "분포 전체"가 같은지를 비모수적으로 검정할 수 있어,
#             정규성을 가정하지 않고도 설비 간 공정조건 차이를 정량화할 수 있음
rows = []
for col in feature_cols:
    stat, p = ks_2samp(unlabeled_cn7[col], unlabeled_rg3[col])
    rows.append({"variable": col, "ks_stat": stat, "p_value": p})

ks_result = pd.DataFrame(rows).sort_values("ks_stat", ascending=False)
print("설비 간(CN7 vs RG3) 분포 차이가 큰 변수 상위 8개 (ks_stat 클수록 분포 상이):")
display(ks_result.head(8))

n_diff = (ks_result["p_value"] < 0.05).sum()
print(f"\n24개 변수 중 {n_diff}개 변수가 p<0.05로 두 설비 간 유의한 분포 차이를 보임")
print("-> 두 설비는 서로 다른 공정 특성을 가지므로, 통합모델과 설비별 모델을 모두 비교해본다.")

## 2. 전처리 및 검증전략

**결정 사항과 근거**
1. **중복행 제거 (분할 전에 수행)**: 1번에서 확인한 대로 라벨 데이터의 절반가량이
   중복행이다. 분할 전에 제거하지 않으면 같은 행이 train/test 양쪽에 들어가는 leakage로
   성능이 과대평가된다. 따라서 **가장 먼저** `drop_duplicates()`를 적용한다.
2. **무분산 변수 제거**: `Clamp_Open_Position`처럼 표준편차가 0인 변수는 드롭한다.
3. **스케일링 생략**: 변수들이 이미 표준화(mean≈0, std≈1)되어 있으므로
   `StandardScaler`를 추가로 적용하지 않는다.
4. **CN7 + RG3 통합 데이터셋 구성, `machine`을 피처로 포함**: 완전히 분리된 모델보다
   통합 학습이 표본 수(중복 제거 후에도 양성 20여 건)를 늘려 불균형 문제를 완화하는 데 유리하다.
   설비 간 분포 차이는 KS 검정에서 확인했으므로, `machine` 원-핫 변수로 그 차이를
   모델이 직접 학습하게 한다.
5. **Stratified split**: 양성 비율이 매우 낮으므로 단순 랜덤 분할은 test set에
   양성 표본이 거의 남지 않을 위험이 있다. `train_test_split(..., stratify=y)`로
   train/test 모두 같은 양성비율을 유지한다.
6. **Test size는 25%, 나머지는 train**: 별도 validation은 두지 않고 교차검증(CV)으로
   대체한다 (표본이 작아 3-way split은 각 분할의 표본 수가 너무 작아짐).

In [ ]:
# 1) 설비별로 먼저 중복 제거 (leakage 방지) -> 2) 무분산 변수 제거 -> 3) 통합 -> 4) 분할
# 왜 설비별로 먼저 제거하나: CN7/RG3는 서로 다른 설비이므로 같은 피처값이라도 우연의 일치일 뿐
#     서로 다른 레코드일 수 있다. 통합 후 중복 제거하면 이런 우연의 일치까지 지워버릴 위험이 있어,
#     설비 내부에서만 완전 동일한 행(피처+라벨)을 중복으로 간주해 제거한다.
labeled_cn7_dedup = labeled_cn7.drop_duplicates()
labeled_rg3_dedup = labeled_rg3.drop_duplicates()
print(f"CN7: {len(labeled_cn7)} -> {len(labeled_cn7_dedup)}행 (중복 {len(labeled_cn7)-len(labeled_cn7_dedup)}건 제거)")
print(f"RG3: {len(labeled_rg3)} -> {len(labeled_rg3_dedup)}행 (중복 {len(labeled_rg3)-len(labeled_rg3_dedup)}건 제거)")

combined = pd.concat([labeled_cn7_dedup, labeled_rg3_dedup], axis=0, ignore_index=True)
combined = combined.drop(columns=zero_var_cols)  # 무분산 변수 제거
combined = pd.get_dummies(combined, columns=["machine"], drop_first=True)  # machine_RG3 (0/1)

X = combined.drop(columns=["PassOrFail"])
y = combined["PassOrFail"]
# 왜 astype(float): get_dummies가 만든 machine_RG3 컬럼은 dtype=bool이다.
# float 컬럼과 bool 컬럼이 섞인 DataFrame을 그대로 두면 SHAP LinearExplainer가
# 내부적으로 dtype=object 배열을 만들어 summary_plot에서 타입 에러가 난다.
X = X.astype(float)
print(f"\n중복제거 후 통합 데이터: {X.shape}, 양성비율={y.mean()*100:.2f}%  (양성 {y.sum()}건)")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

print(f"train: {X_train.shape}, 양성비율={y_train.mean()*100:.2f}%  (양성 {y_train.sum()}건)")
print(f"test : {X_test.shape}, 양성비율={y_test.mean()*100:.2f}%  (양성 {y_test.sum()}건)")

## 3. AI 예측모델 개발 (서면평가 2번, 40점 — 배점 최대 항목)

**요구내용**: 제조 이상 확률을 예측하는 모델 개발, 베이스라인 포함 2개 이상 모델 비교
(F1-score 등), 최종모델 선정근거 설명.

**왜 정확도(accuracy)를 쓰지 않나**: 양성비율이 1.4~2.1%이므로 "전부 정상"이라고만 예측해도
accuracy는 98%를 넘는다. 이 과제에서 accuracy는 사실상 의미가 없으므로,
**F1-score, PR-AUC(average precision), ROC-AUC**를 주 지표로 사용한다.
특히 PR-AUC는 불균형 데이터에서 양성 클래스 탐지력을 가장 잘 보여주는 지표로 알려져 있다.

**모델 3종 비교 계획**
1. **로지스틱회귀 (베이스라인)**: `class_weight="balanced"`로 불균형 보정. 해석이 쉬워
   기준선(baseline) 역할에 적합.
2. **RandomForest**: 비선형/변수 간 상호작용을 잡아내며, `class_weight="balanced"`도 지원.
3. **XGBoost**: 불균형 데이터에서 강건한 성능을 보이는 GBM 계열. `scale_pos_weight`로
   불균형을 보정하고, 표 형태 데이터에서 SOTA급 성능을 내는 경우가 많아 최종모델 후보 1순위.

In [ ]:
def evaluate(name, y_te, y_proba, threshold=0.5):
    """공통 평가 함수: F1, PR-AUC, ROC-AUC를 한번에 계산.
    왜 함수로 분리: 3개 모델에 동일한 평가기준을 일관되게 적용하기 위함 (재현성/비교 공정성)."""
    y_pred = (y_proba >= threshold).astype(int)
    return {
        "model": name,
        f"f1@{threshold}": f1_score(y_te, y_pred),
        "pr_auc": average_precision_score(y_te, y_proba),
        "roc_auc": roc_auc_score(y_te, y_proba),
    }

results = []

# 1) 베이스라인: 로지스틱회귀
logreg = LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE)
logreg.fit(X_train, y_train)
proba_lr = logreg.predict_proba(X_test)[:, 1]
results.append(evaluate("LogisticRegression (baseline)", y_test, proba_lr))

# 2) RandomForest
rf = RandomForestClassifier(
    n_estimators=400, max_depth=6, class_weight="balanced",
    random_state=RANDOM_STATE, n_jobs=-1,
)
rf.fit(X_train, y_train)
proba_rf = rf.predict_proba(X_test)[:, 1]
results.append(evaluate("RandomForest", y_test, proba_rf))

# 3) XGBoost - scale_pos_weight = 음성수/양성수 로 불균형 보정 (공식 권장 방식)
spw = (y_train == 0).sum() / (y_train == 1).sum()
xgb_clf = xgb.XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    scale_pos_weight=spw, eval_metric="aucpr",
    random_state=RANDOM_STATE, n_jobs=-1,
)
xgb_clf.fit(X_train, y_train)
proba_xgb = xgb_clf.predict_proba(X_test)[:, 1]
results.append(evaluate("XGBoost", y_test, proba_xgb))

results_df = pd.DataFrame(results).round(3)
display(results_df)

In [ ]:
# Precision-Recall 곡선 비교
# 왜 ROC 대신 PR을 메인 그래프로: 양성비율이 매우 낮을 때 ROC-AUC는 낙관적으로 보이는 경향이
#     있어(음성이 압도적으로 많아 FPR이 잘 안 오름), PR 곡선이 실제 검사 현장에서 중요한
#     "양성 예측의 정밀도-재현율 트레이드오프"를 더 직접적으로 보여준다.
fig, ax = plt.subplots(figsize=(7, 6))
for name, proba in [("LogisticRegression", proba_lr), ("RandomForest", proba_rf), ("XGBoost", proba_xgb)]:
    PrecisionRecallDisplay.from_predictions(y_test, proba, name=name, ax=ax)
ax.set_title("Precision-Recall Curve 비교 (test set)")
plt.tight_layout()
plt.show()

In [ ]:
# 임계값(threshold) 튜닝: F1을 최대화하는 지점을 탐색
# 왜: 기본 임계값 0.5는 불균형 데이터에서 최적이 아닌 경우가 대부분. 검사 우선순위 결정에
#     쓰일 최종 임계값은 F1이 최대가 되는 지점을 기준점으로 삼고, 이후 4번(현장 활용방안)에서
#     "전수검사/샘플검사" 등급을 나눌 때 재사용한다.
def best_threshold(y_te, proba):
    prec, rec, thr = precision_recall_curve(y_te, proba)
    f1s = 2 * prec * rec / (prec + rec + 1e-9)
    best_idx = np.nanargmax(f1s[:-1])  # thr 길이가 prec/rec보다 1 짧음
    return thr[best_idx], f1s[best_idx]

for name, proba in [("LogisticRegression", proba_lr), ("RandomForest", proba_rf), ("XGBoost", proba_xgb)]:
    t, f1 = best_threshold(y_test, proba)
    print(f"{name:24s} 최적임계값={t:.3f}  F1@최적={f1:.3f}")

### 3-1. 단일 split 평가의 한계 재점검 — 교차검증 재평가

**문제의식**: 위 표에서는 세 모델의 PR-AUC가 전부 0.05 안팎으로 낮고 비슷했다. 그런데
test set에는 양성이 10건뿐이다 — 어떤 10건이 test에 뽑히느냐에 따라 점수가 크게 출렁이는
**고분산 추정치**라는 뜻이다. 표본이 극소수일 때 단일 split 평가만으로 "이게 최선"이라고
단정하는 것은 위험하다. 그래서 **반복 계층화 교차검증(Repeated Stratified K-Fold)**으로
다시 평가해, 데이터를 다양하게 나눠본 평균 성능을 확인한다.

In [ ]:
# 왜 5-fold x 10회 반복인가: 양성표본 39건을 5등분하면 fold당 양성 7~8건뿐이라
# 한 번의 5-fold만으로도 분산이 크다. 10회 반복(총 50개 추정치)의 평균을 보면
# 특정 분할의 운에 의존하지 않는 더 신뢰도 높은 비교가 가능하다.
rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE)
pr_auc_scorer = make_scorer(average_precision_score, response_method="predict_proba")

cv_results = []
for name, model in [
    ("LogisticRegression (C=1.0, 기본값)",
     LogisticRegression(class_weight="balanced", max_iter=3000, random_state=RANDOM_STATE)),
    ("RandomForest",
     RandomForestClassifier(n_estimators=400, max_depth=6, class_weight="balanced",
                             random_state=RANDOM_STATE, n_jobs=-1)),
    ("XGBoost",
     xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, scale_pos_weight=spw,
                        eval_metric="aucpr", random_state=RANDOM_STATE, n_jobs=-1)),
]:
    scores = cross_val_score(model, X, y, cv=rcv, scoring=pr_auc_scorer, n_jobs=-1)
    cv_results.append({"model": name, "cv_pr_auc_mean": scores.mean(), "cv_pr_auc_std": scores.std()})

cv_results_df = pd.DataFrame(cv_results).round(3)
display(cv_results_df)

**결과**: 교차검증으로 다시 보면 트리 앙상블(RandomForest, XGBoost)보다
**로지스틱회귀가 뚜렷하게 우세**하다. 표본(양성 29~31건 수준)이 극히 적을 때는 트리 기반
모델이 노이즈까지 학습해 과적합하기 쉬운 반면, 파라미터 수가 적은 선형모델은 편향(bias)은
크지만 분산(variance)이 낮아 오히려 더 안정적으로 일반화된다 — 전형적인 편향-분산 트레이드오프
사례다. 이 발견에 따라 로지스틱회귀의 규제강도(C)를 교차검증으로 튜닝해 최종모델을 다시 정한다.

In [ ]:
# 규제강도(C) 튜닝: C가 작을수록 규제가 강해져(계수를 0에 가깝게 억제) 과적합을 억제한다.
# 표본이 적을수록 강한 규제가 유리할 것이라는 가설을 CV로 직접 검증한다.
c_grid = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
c_scores = []
for C in c_grid:
    model = LogisticRegression(C=C, class_weight="balanced", max_iter=3000, random_state=RANDOM_STATE)
    scores = cross_val_score(model, X, y, cv=rcv, scoring=pr_auc_scorer, n_jobs=-1)
    c_scores.append({"C": C, "cv_pr_auc_mean": scores.mean(), "cv_pr_auc_std": scores.std()})

c_scores_df = pd.DataFrame(c_scores)
display(c_scores_df)

best_C = float(c_scores_df.loc[c_scores_df["cv_pr_auc_mean"].idxmax(), "C"])
print(f"\nCV 기준 최적 C = {best_C}")

# 최종모델 확정: 튜닝된 로지스틱회귀. 이후 SHAP/오류분석/보정/검사우선순위 단계는
# 앞서 만든 X_train/X_test(stratified split)를 그대로 재사용해 일관성을 유지한다.
final_model = LogisticRegression(C=best_C, class_weight="balanced", max_iter=3000, random_state=RANDOM_STATE)
final_model.fit(X_train, y_train)
proba_final = final_model.predict_proba(X_test)[:, 1]
print("final_model held-out test PR-AUC:", round(average_precision_score(y_test, proba_final), 3))

### 최종모델 선정근거

- **최종모델: 규제 튜닝된 로지스틱회귀(`final_model`)**. 반복 교차검증에서 RandomForest,
  XGBoost보다 일관되게 높은 PR-AUC를 보였고, C 튜닝으로 한 번 더 개선을 확인했다 (위 결과).
- RandomForest·XGBoost는 여전히 "비교모델"로 유지한다 — 서면평가 2번이 요구하는
  "베이스라인 포함 2개 이상 모델 비교"를 만족시키고, 트리 모델이 왜 이 데이터에서
  불리한지(표본 극소 → 과적합) 자체가 3번(영향요인·오류분석)에 쓸 수 있는 진단 결과다.
- **다른 개선 방법도 시도했으나 채택하지 않은 것들** (근거를 남기는 것도 평가에 도움이 됨):
  - *SMOTE 오버샘플링*: 교차검증 파이프라인 안에서 시도했으나 로지스틱회귀 성능을 오히려
    떨어뜨렸다. 양성 29건으로 보간을 하면 실제로 존재하지 않는 가짜 경계를 만들어낼
    위험이 실증적으로 확인된 셈이다.
  - *IsolationForest 이상치점수를 피처로 추가*: 대량 unlabeled 데이터로 학습한 이상탐지
    점수를 추가 피처로 넣어봤으나 성능 변화가 거의 없었다. 원인은 이상탐지가 포착하는
    "통계적 이상치"와 실제 "불량"이 반드시 일치하지 않기 때문으로 보인다 (결론에서 후속 과제로 제안).
- 임계값은 0.5 고정이 아니라, `proba_final` 기준으로 새로 구한 F1 최적 임계값을 사용한다.
  6번(현장 활용방안)에서는 **보정된 확률(`proba_cal`)에 대해 별도로 재계산한 임계값**을
  사용한다 — 보정 전후 확률의 스케일이 다르므로 서로 다른 척도의 임계값을 섞어 쓰면 안 된다.

## 4. 영향요인 및 오류분석 (서면평가 3번, 15점)

**요구내용**: 주요 영향변수와 변수 간 상호작용 분석, False Negative/False Positive가
집중되는 공정조건 도출.

**왜 SHAP인가**: 최종모델이 로지스틱회귀로 바뀌었지만, 계수(coefficient)만으로는 변수들이
서로 다른 스케일/상관관계를 가질 때 상대적 영향력을 직관적으로 비교하기 어렵다. `shap`의
`LinearExplainer`는 선형모델에 최적화된 정확한 Shapley value를 제공해, 개별 예측 단위로
"이 변수가 불량 확률을 얼마나 올렸는지/내렸는지"까지 설명할 수 있어 서면평가 요구사항
(주요 영향변수 + 상호작용)에 정확히 부합한다. (참고: RandomForest/XGBoost 비교모델을 최종
채택했다면 `TreeExplainer`를 썼겠지만, 최종모델이 선형모델이므로 그에 맞는 explainer를 쓴다.)

In [ ]:
# SHAP 값 계산 (LinearExplainer: 로지스틱회귀 등 선형모델 전용, 정확하고 빠름)
explainer = shap.LinearExplainer(final_model, X_train)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
plt.title("변수 중요도 (mean |SHAP value|)")
plt.tight_layout()
plt.show()

shap.summary_plot(shap_values, X_test, show=False)
plt.title("SHAP summary (방향성 포함)")
plt.tight_layout()
plt.show()

In [ ]:
# False Negative / False Positive가 집중되는 공정조건 분석
# 왜: 단순히 "정확도가 낮다"가 아니라, "어떤 공정조건에서 모델이 틀리는가"를 밝혀야
#     서면평가 3번 배점을 확보할 수 있고, 4번(현장 활용방안)의 현장 대응책과도 연결된다.
best_t, _ = best_threshold(y_test, proba_final)
y_pred_final = (proba_final >= best_t).astype(int)

error_df = X_test.copy()
error_df["y_true"] = y_test.values
error_df["y_pred"] = y_pred_final
error_df["proba"] = proba_final

def tag(row):
    if row.y_true == 1 and row.y_pred == 0:
        return "FN"
    if row.y_true == 0 and row.y_pred == 1:
        return "FP"
    if row.y_true == 1 and row.y_pred == 1:
        return "TP"
    return "TN"

error_df["type"] = error_df.apply(tag, axis=1)
print(error_df["type"].value_counts())

top_shap_features = (
    pd.DataFrame(np.abs(shap_values), columns=X_test.columns).mean().sort_values(ascending=False).head(5).index.tolist()
)
print(f"\nSHAP 상위 5개 변수 기준 FN/FP 구간 비교: {top_shap_features}")
display(error_df.groupby("type")[top_shap_features].mean().round(2))

## 5. 창의성·차별성 (서면평가 5번, 10점)

**요구내용**: 불균형 학습, 앙상블, 불확실성 추정, 확률보정, 제조지식 결합 등 차별화된 방법 제안.

**이 노트북에서 적용한 차별화 요소**
1. **불균형 학습**: `class_weight`/`scale_pos_weight` (위 3번에서 이미 적용) — 데이터를
   합성 증강(SMOTE)하는 대신 손실 가중치 조정 방식을 우선 채택. 근거: 양성 표본이 17~25건으로
   극소수라 SMOTE로 보간하면 실제 존재하지 않는 가짜 패턴을 만들어낼 위험이 크기 때문.
2. **앙상블**: RandomForest + XGBoost 두 트리 앙상블을 비교 (3번).
3. **확률보정(calibration)**: 아래에서 `CalibratedClassifierCV`로 예측확률의 신뢰도를 높인다.
   왜 필요한가: 불균형 보정을 위해 `scale_pos_weight`를 쓰면 예측확률이 실제 사건 발생확률보다
   과장되게(또는 왜곡되게) 나오는 경우가 많다. 검사 우선순위처럼 "확률값 자체"를 실무 의사결정에
   쓰려면(예: 상위 10% 위험군 선별) 보정된 확률이 필요하다.

In [ ]:
# 확률보정: sigmoid(Platt scaling) 방식. 표본이 매우 작을 때 isotonic은 유연한 만큼
# 과적합 위험이 크므로, 파라미터가 2개뿐인 sigmoid 방식이 이 표본 크기에는 더 안전하다.
# 왜 cv=5: 표본이 작으므로 내부 교차검증으로 보정곡선을 안정화한다.
cal_sigmoid = CalibratedClassifierCV(final_model, method="sigmoid", cv=5)
cal_sigmoid.fit(X_train, y_train)
proba_cal = cal_sigmoid.predict_proba(X_test)[:, 1]

print("보정 전 PR-AUC :", round(average_precision_score(y_test, proba_final), 3))
print("보정 후 PR-AUC :", round(average_precision_score(y_test, proba_cal), 3))

# 왜 별도 임계값이 필요한가: 보정(calibration)은 확률의 "스케일"을 바꾸므로,
# 보정 전 확률(proba_xgb)로 구한 best_t를 보정 후 확률(proba_cal)에 그대로 쓰면 안 된다.
# 6번(현장 활용방안)에서는 반드시 proba_cal 기준으로 새로 계산한 임계값을 사용한다.
best_t_cal, best_f1_cal = best_threshold(y_test, proba_cal)
print(f"\n보정된 확률 기준 최적임계값={best_t_cal:.3f}  F1@최적={best_f1_cal:.3f}")

# 불확실성 추정: RandomForest는 트리별 예측확률의 분산으로 예측 신뢰구간을 근사할 수 있음
# 왜: 검사 우선순위를 정할 때 "확률은 낮지만 트리마다 의견이 갈리는(불확실한) 샘플"은
#     별도로 재검토 대상에 포함시키는 것이 안전 측면에서 합리적이다.
tree_probas = np.stack([t.predict_proba(X_test)[:, 1] for t in rf.estimators_], axis=0)
proba_std = tree_probas.std(axis=0)

uncertainty_df = pd.DataFrame({
    "proba_mean": tree_probas.mean(axis=0),
    "proba_std": proba_std,
}, index=X_test.index).sort_values("proba_std", ascending=False)
print("\n트리 간 예측 분산(불확실성)이 큰 상위 5개 샘플:")
display(uncertainty_df.head())

## 6. 현장 활용방안 (서면평가 4번, 10점)

**요구내용**: 예측결과를 사전경보, 품질검사 우선순위, 공정점검 또는 작업자 의사결정에
활용하는 방안 제안.

**제안: 3단계 검사 우선순위 등급**
보정된 예측확률(`proba_cal`)을 기준으로 등급을 나눈다. 왜 3단계인가: 이분법(합격/불합격)
대신 등급을 두면, 현장에서 전수검사 인력을 "위험도가 높은 제품"에 집중 배치할 수 있어
검사비용 대비 불량 탐지율을 높일 수 있다 (출제 배경의 "검사 우선순위 결정" 요구사항과 직결).

- **고위험(즉시 전수검사)**: 예측확률 상위 구간 — 보정확률 기준 F1 최적임계값(`best_t_cal`) 이상
- **주의(샘플검사 강화)**: 중간 구간 — 최적임계값의 절반 이상 ~ 최적임계값 미만
- **정상(기존 절차 유지)**: 하위 구간

주의: 4번(영향요인·오류분석)에서 쓴 `best_t`는 **보정 전** `proba_xgb` 기준 임계값이고,
여기서 쓰는 `best_t_cal`은 **보정 후** `proba_cal` 기준 임계값이다. 두 확률은 스케일이
다르므로 반드시 각자의 임계값과 짝을 맞춰 사용한다.

In [ ]:
def inspection_priority(proba, threshold, low_ratio=0.5):
    """보정된 예측확률을 3단계 검사 등급으로 변환.
    왜 최적임계값의 절반을 '주의' 하한선으로 쓰나: 임계값 근방의 샘플은 모델이 확신하지
    못하는 경계 영역이므로, 완전히 정상으로 흘려보내지 않고 샘플검사를 강화하는
    완충 구간을 두기 위함."""
    grade = np.where(proba >= threshold, "고위험(전수검사)",
             np.where(proba >= threshold * low_ratio, "주의(샘플검사 강화)", "정상(기존 절차 유지)"))
    return grade

priority = inspection_priority(proba_cal, best_t_cal)
priority_df = pd.DataFrame({"proba_cal": proba_cal, "grade": priority, "y_true": y_test.values})

print("검사 등급별 분포 및 실제 불량 검출률:")
display(
    priority_df.groupby("grade").agg(
        건수=("y_true", "size"),
        실제불량건수=("y_true", "sum"),
        불량비율=("y_true", "mean"),
    ).round(3)
)

## 7. 코드 및 재현성 (서면평가 6번, 10점)

**요구내용**: 동일 환경에서 데이터 전처리부터 학습·추론·결과생성까지 자동 실행.

**적용한 재현성 장치**
- `RANDOM_STATE = 42`를 모든 확률적 요소(분할, 모델 초기화)에 일괄 적용
- 모든 파일 경로를 상대경로(`DATA_DIR = "."`)로 지정해 제출 zip 구조 그대로 실행 가능
- 아래처럼 `requirements.txt`를 별도 생성해 동일 패키지 버전을 재현 가능하게 한다
- 노트북을 위에서 아래로 `Run All` 하면 데이터 로드 -> EDA -> 전처리 -> 3개 모델 학습 ->
  평가 -> SHAP 분석 -> 오류분석 -> 확률보정 -> 검사우선순위표 생성까지 자동 실행된다.

In [ ]:
# requirements.txt 생성 (제출용 zip에 포함)
# 왜: 서면평가 6번이 "동일 환경에서 자동 실행"을 요구하므로, 설치 버전을 명시해야
#     심사위원 환경에서도 동일한 결과가 재현된다.
import sklearn
requirements = f"""pandas=={pd.__version__}
numpy=={np.__version__}
scikit-learn=={sklearn.__version__}
xgboost=={xgb.__version__}
shap=={shap.__version__}
matplotlib=={plt.matplotlib.__version__}
seaborn=={sns.__version__}
"""
with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements)
print(requirements)

## 결론 및 다음 단계

- CN7/RG3 두 사출성형기의 공정데이터를 통합 학습하여, **규제 튜닝된 로지스틱회귀**로
  극소수(중복 제거 후 3.16%) 불량 클래스를 예측하는 모델을 구축했다.
- 단일 split 평가는 표본이 적을 때 신뢰할 수 없다는 것을 반복 교차검증으로 직접 확인했고,
  그 결과 애초 유력 후보였던 트리 앙상블(RF/XGBoost) 대신 더 단순한 선형모델이 실제로
  더 나은 일반화 성능을 낸다는, 데이터 규모에 따른 모델 선택의 반례를 보고서에 남길 수 있었다.
- SHAP(LinearExplainer) 분석으로 주요 영향변수를 식별하고, FN/FP가 집중되는 공정조건을
  도출했으며, 보정된 예측확률을 3단계 검사 우선순위로 변환해 현장 활용방안을 제시했다.
- SMOTE, IsolationForest 이상치점수 결합 등 추가 개선 방법을 시도했으나 채택하지 않았고,
  그 이유(가짜 패턴 생성 위험, 이상치≠불량 불일치)를 근거와 함께 남겼다.

**다음 단계(시간이 허락하면 추가로 발전시킬 부분)**
1. **가장 우선순위 높은 개선**: 대량의 `unlabeled` 데이터(각 설비 3만5천여 건)를 활용해
   준지도학습(pseudo-labeling)이나 오토인코더 재구성오차 기반 이상탐지를 라벨 확장 수단으로
   결합 — 현재 성능을 제한하는 근본 원인(양성표본 30여 건)에 직접 대응하는 방향이다.
2. 설비별(CN7 전용, RG3 전용) 모델과 통합모델의 성능을 정량 비교하는 실험 추가.
3. 발표자료용으로 SHAP dependence plot을 상위 2~3개 변수에 대해 추가 시각화.
4. 로지스틱회귀의 규제 경로(regularization path)를 L1으로도 튜닝해 변수 선택 결과를
   RF/XGBoost의 변수중요도와 교차검증(어떤 변수가 일관되게 중요한지 확인).